# Messi goals: PCA and t-SNE

Explore 704 recorded goals. Questions: what structure appears in a mixed context feature space; how do linear variance and local-neighborhood views differ; can records be linked? Run from repository root with `python -m nbconvert --execute --to notebook --inplace final_dimension_reduction.ipynb`.

## 1–3. Provenance, schema, cleaning, feature engineering

The source is `data/messi_goals.csv`; existing project notebooks informed season and stoppage-time cleanup patterns. A stable `goal_id` is assigned from source row order before filtering. Standardized minute and season start year are combined with one-hot competition, venue, position, type, and opponent. One-hot mismatch is suitable for nominal values; opponents occurring fewer than five times are pooled. Assists are excluded (215 missing and 69 distinct names) to avoid sparse identity geometry. Matchday, date, club, result, and score are omitted as identifiers, redundant context, or outcome-like fields.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from src.data_processing import load_goals, prepare_features
from src.dr_analysis import project
ROOT = Path.cwd()
if not (ROOT / 'data/messi_goals.csv').exists(): ROOT = ROOT.parent
raw = load_goals(ROOT / 'data/messi_goals.csv')
clean, X, prep = prepare_features(raw)
print('Rows/columns:', len(raw), len(raw.columns)-1)
print('Missing:', raw.drop(columns='goal_id').isna().sum().to_dict())
print('Cardinality:', raw.select_dtypes(include='string').nunique().to_dict())
print('Unparsed minutes:', int(clean.minute_num.isna().sum()), 'unparsed dates:', int(clean.date_parsed.isna().sum()), 'matrix shape:', X.shape)
clean.head()

Rows/columns: 704 13
Missing: {'Season': 0, 'Competition': 0, 'Matchday': 0, 'Date': 0, 'Venue': 0, 'Club': 0, 'Opponent': 0, 'Result': 0, 'Playing_Position': 0, 'Minute': 0, 'At_score': 0, 'Type': 1, 'Goal_assist': 215}
Cardinality: {'goal_id': 704, 'Season': 19, 'Competition': 8, 'Matchday': 48, 'Date': 455, 'Venue': 2, 'Club': 2, 'Opponent': 98, 'Result': 55, 'Playing_Position': 5, 'Minute': 97, 'At_score': 46, 'Type': 12, 'Goal_assist': 70}
Unparsed minutes: 0 unparsed dates: 0 matrix shape: (704, 61)


,goal_id,Season,Competition,Matchday,Date,Venue,Club,Opponent,Result,Playing_Position,Minute,At_score,Type,Goal_assist,minute_num,season_start,date_parsed
0,goal-0001,04/05,LaLiga,34,05-01/05,H,FC Barcelona,Albacete Balompie,2:00,CF,90+1,2:00,Left-footed shot,Ronaldinho Gaacho,91.0,2004.0,2005-01-05 00:00:00
1,goal-0002,05/06,UEFA Champions League,Group Stage,11-02/05,H,FC Barcelona,Panathinaikos Athens,5:00,RW,34,3:00,Left-footed shot,<NA>,34.0,2005.0,2005-02-11 00:00:00
2,goal-0003,05/06,LaLiga,13,11/27/05,H,FC Barcelona,Racing Santander,4:01,RW,51,2:00,Left-footed shot,Samuel Etoo,51.0,2005.0,2005-11-27 00:00:00
3,goal-0004,05/06,LaLiga,19,1/15/06,H,FC Barcelona,Athletic Bilbao,2:01,RW,50,2:01,Left-footed shot,Mark van Bommel,50.0,2005.0,2006-01-15 00:00:00
4,goal-0005,05/06,LaLiga,20,1/22/06,H,FC Barcelona,Deportivo Alaves,2:00,CF,82,2:00,Left-footed shot,Ronaldinho Gaacho,82.0,2005.0,2006-01-22 00:00:00


## 4. Original high-dimensional data views

Distributions and numeric parallel coordinates provide interpretable views before projection; the compact parallel view is not a replacement for the full encoded matrix.

In [2]:
display(raw['Competition'].value_counts().to_frame('goals'))
px.histogram(clean, x='minute_num', nbins=20, title='Goal minutes').show()
px.bar(clean['Type'].fillna('Unknown').value_counts().rename_axis('Type').reset_index(name='Goals'), x='Type', y='Goals', title='Goal types').show()
px.parallel_coordinates(clean, dimensions=['season_start','minute_num'], labels={'season_start':'Season starting year','minute_num':'Minute'}, title='Numeric original-space profile').show()

,goals
Competition,
LaLiga,474
UEFA Champions League,129
Copa del Rey,56
Ligue 1,21
Supercopa,14
FIFA Club World Cup,5
UEFA Super Cup,3
Trophée des Champions,2


## 5. PCA

PCA maximizes retained linear variance along orthogonal axes. Two-component variance is reported from the fitted matrix; this does not guarantee local neighbor preservation.

## 6. t-SNE

t-SNE emphasizes local similarities. Perplexity is constrained below sample count and random_state is fixed. Island spacing is not a global distance, and apparent clusters are not independent classes.

In [3]:
coords, metrics = project(X, clean)
print(metrics)
coords[['goal_id','PC1','PC2','TSNE1','TSNE2','Season','Competition','Opponent','Minute','Type']].to_csv(ROOT/'goal_projections.csv', index=False)
coords.head()

{'pca_variance': [0.20706541674989884, 0.1924592319458436], 'pca_variance_total': 0.39952464869574245, 'pca_trustworthiness': 0.8846417607447019, 'tsne_trustworthiness': 0.9753789941902687, 'trustworthiness_neighbors': 10, 'tsne_perplexity': 30.0}


,goal_id,Season,Competition,Matchday,Date,Venue,Club,Opponent,Result,Playing_Position,...,At_score,Type,Goal_assist,minute_num,season_start,date_parsed,PC1,PC2,TSNE1,TSNE2
0,goal-0001,04/05,LaLiga,34,05-01/05,H,FC Barcelona,Albacete Balompie,2:00,CF,...,2:00,Left-footed shot,Ronaldinho Gaacho,91.0,2004.0,2005-01-05 00:00:00,-2.650875,-0.074248,4.116431,-30.645832
1,goal-0002,05/06,UEFA Champions League,Group Stage,11-02/05,H,FC Barcelona,Panathinaikos Athens,5:00,RW,...,3:00,Left-footed shot,<NA>,34.0,2005.0,2005-02-11 00:00:00,-1.110470,-1.712015,-27.171024,-1.983637
2,goal-0003,05/06,LaLiga,13,11/27/05,H,FC Barcelona,Racing Santander,4:01,RW,...,2:00,Left-footed shot,Samuel Etoo,51.0,2005.0,2005-11-27 00:00:00,-1.629493,-1.235395,-33.889549,-2.315434
3,goal-0004,05/06,LaLiga,19,1/15/06,H,FC Barcelona,Athletic Bilbao,2:01,RW,...,2:01,Left-footed shot,Mark van Bommel,50.0,2005.0,2006-01-15 00:00:00,-1.593797,-1.255987,-33.292496,-1.786484
4,goal-0005,05/06,LaLiga,20,1/22/06,H,FC Barcelona,Deportivo Alaves,2:00,CF,...,2:00,Left-footed shot,Ronaldinho Gaacho,82.0,2005.0,2006-01-22 00:00:00,-2.311570,-0.253844,4.159688,-30.573956


## 7–8. Linked exploration and comparison

The Streamlit dashboard synchronizes a multiselect of stable goal IDs across both projections, original-feature summaries, and the detail table. Trustworthiness compares neighborhood preservation at a common valid k.

In [4]:
px.scatter(coords, x='PC1', y='PC2', color='Type', hover_data=['goal_id','Season','Opponent','Competition','Minute'], title='PCA').show()
px.scatter(coords, x='TSNE1', y='TSNE2', color='Type', hover_data=['goal_id','Season','Opponent','Competition','Minute'], title='t-SNE').show()
print(f"PCA retains {metrics['pca_variance_total']:.1%} variance; k={metrics['trustworthiness_neighbors']}: PCA trust={metrics['pca_trustworthiness']:.3f}, t-SNE trust={metrics['tsne_trustworthiness']:.3f}")

PCA retains 40.0% variance; k=10: PCA trust=0.885, t-SNE trust=0.975


## 10–11. Limitations, conclusion, reproducibility

This single-player event log has uneven categories, missing assists, and no tracking coordinates. One-hot encoding defines a chosen mismatch geometry. PCA is global and linear; t-SNE is local and parameter-sensitive. Neither supports causal claims. Together they provide complementary views. Reproduce from repo root with `python -m pip install -r requirements.txt`, execute this notebook, or launch `streamlit run app.py`.

## 12. Approved bonus: context projection matrix

The dashboard now contains a 3x3 matrix of PCA projections over match, goal, and career context groups. Each cell projects one group or a pair of groups; selecting a cell enlarges it. Selected goal IDs remain highlighted in the matrix and the main linked views. This adapts the Dimension Projection Matrix idea in Yuan et al. (2013) but does not implement their projection tree or full interaction model.